# 🚀 KTRS 마케팅 봇 전용 구글 무료 GPU (RealVisXL 극실사 AI 이미지 서버)
이 노트북은 **Google Colab 무료 GPU(Tesla T4)**를 활용하여 **비용 0원으로 극실사 4K 인물 사진을 무제한 생성**하는 서버입니다.

### 📌 사용 방법:
1. 상단 메뉴 **[런타임 ➔ 모두 실행 (Ctrl+F9)]** 클릭
2. 1분 후 아래에 출력되는 `COLAB_GPU_API_URL`을 복사하면 끝!

In [ ]:
# 1. 필수 라이브러리 및 Cloudflare 터널 자동 설치
!pip install -q diffusers transformers accelerate safetensors fastapi uvicorn pillow pydantic
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 2. RealVisXL V4.0 극실사 AI 모델 로딩 & 백그라운드 서버 가동
import io
import os
import re
import time
import base64
import random
import threading
import subprocess
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
from PIL import Image
import uvicorn
from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler

app = FastAPI(title="KTRS RealVisXL GPU Image Server")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 가동 GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

# 모델 로드
MODEL_ID = "SG161222/RealVisXL_V4.0"
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe = pipe.to(device)
if device == "cuda":
    pipe.enable_attention_slicing()
print("✅ RealVisXL 극실사 AI 모델 로딩 완료!")

class ImageGenRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "caucasian, white, deformed fingers, extra limbs, bad anatomy, ugly, blurry, 3d render, cartoon, plastic skin"
    aspect_ratio: Optional[str] = "9:16"
    seed: Optional[int] = -1
    guidance_scale: Optional[float] = 5.0
    num_inference_steps: Optional[int] = 25

@app.get("/")
def health():
    return {"status": "ok", "engine": "RealVisXL V4.0", "device": device}

@app.post("/generate")
def generate_image(req: ImageGenRequest):
    try:
        if req.aspect_ratio == "9:16":
            width, height = 768, 1344
        elif req.aspect_ratio == "1:1":
            width, height = 1024, 1024
        elif req.aspect_ratio == "16:9":
            width, height = 1344, 768
        else:
            width, height = 768, 1344

        used_seed = req.seed if req.seed is not None and req.seed >= 0 else random.randint(100000, 999999999)
        generator = torch.Generator(device=device).manual_seed(used_seed)

        image = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            width=width,
            height=height,
            guidance_scale=req.guidance_scale,
            num_inference_steps=req.num_inference_steps,
            generator=generator
        ).images[0]

        buffered = io.BytesIO()
        image.save(buffered, format="JPEG", quality=95)
        img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")

        return {"success": True, "seed": used_seed, "image_base64": img_str, "width": width, "height": height}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 1. 백그라운드 스레드로 FastAPI 서버 가동 (Jupyter 충돌 완전 방지)
def start_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

# 2. Cloudflare 무설정 터널 가동 (토큰 불필요)
proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

public_url = None
for _ in range(30):
    line = proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print(f"\n========================================================")
print(f"🎉 [KTRS 마케팅 봇 연결용 공용 URL 생성 완료!]")
print(f"👉 아래 주소를 복사해 주세요:")
print(f"   COLAB_GPU_API_URL={public_url}")
print(f"========================================================\n")